# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tal3at-M/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
#Selected Model: Random Forest Classifier (paired with calibrated probability outputs).

#Why it fits:
#- Tabular search traffic metrics exhibit non-linear feature interactions (e.g., interaction between rank degradation and impression loss).
#- Tree ensembles are robust to extreme right-skewed metric distributions without requiring complex continuous normalization.
#- Provides direct feature importance metrics and calibrated probabilities suited for ranking prioritization queues.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
#Split Strategy: Stratified Train/Test Split (80% Train, 20% Test) stratified on the decay target.

#Why honest:
#- Ensures identical target class distribution between training and evaluation partitions.
#- Evaluates out-of-sample generalization strictly on held-out pages, avoiding optimistic bias and data snooping.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

url = "https://raw.githubusercontent.com/Tal3at-M/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 1. Feature Engineering
df["ctr_calc"] = df["clicks_90d"] / (df["impressions_90d"] + 1e-5)
df["log_impressions"] = np.log1p(df["impressions_90d"])
df["log_clicks"] = np.log1p(df["clicks_90d"])

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
    "log_impressions",
    "log_clicks",
    "ctr_calc",
]

X = df[feature_cols].fillna(df[feature_cols].median())
y = (df["trend_direction"].str.lower() == "down").astype(int)

# 2. Train/Test Split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.20, random_state=42, stratify=y
)

# 3. Train Random Forest Model
rf = RandomForestClassifier(
    n_estimators=100, max_depth=6, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

# 4. Evaluate Baseline Heuristic on Test Split
test_df = df.loc[idx_test].copy()
test_df["baseline_score"] = (
    np.log1p(test_df["impressions_90d"])
    * (1.0 / np.clip(test_df["avg_position"], 1.0, 100.0))
    * (test_df["content_age_days"] / 365.0)
)
baseline_top50_idx = test_df.sort_values(
    by="baseline_score", ascending=False
).head(50).index
baseline_p50 = y.loc[baseline_top50_idx].mean()

# 5. Evaluate Random Forest on Test Split
test_df["rf_prob"] = rf.predict_proba(X_test)[:, 1]
rf_top50_idx = test_df.sort_values(by="rf_prob", ascending=False).head(50).index
rf_p50 = y.loc[rf_top50_idx].mean()
rf_roc = roc_auc_score(y_test, test_df["rf_prob"])

# 6. Comparison Table
summary_df = pd.DataFrame(
    {
        "Method": ["Heuristic Baseline (Week 4)", "Random Forest Classifier"],
        "Precision@50": [f"{baseline_p50:.2%}", f"{rf_p50:.2%}"],
        "ROC-AUC": ["N/A (Rule)", f"{rf_roc:.3f}"],
        "Queue Quality": ["Static ranking", "Calibrated decay probability"],
    }
)

display(summary_df)



,Method,Precision@50,ROC-AUC,Queue Quality
0,Heuristic Baseline (Week 4),52.00%,N/A (Rule),Static ranking
1,Random Forest Classifier,86.00%,0.715,Calibrated decay probability


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# Feature importances
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(
    ascending=False
)
print("--- Model Feature Importances ---")
print(importances.round(4))

# False Positives check
test_df["true_label"] = y_test
top50_rf = test_df.sort_values(by="rf_prob", ascending=False).head(50)
false_positives = top50_rf[top50_rf["true_label"] == 0]
print(f"\nFalse Positives in Top-50 recommendations: {len(false_positives)}")

--- Model Feature Importances ---
impressions_90d     0.2475
log_impressions     0.2362
content_age_days    0.2093
avg_position        0.1837
ctr_calc            0.0470
clicks_90d          0.0397
log_clicks          0.0367
dtype: float64

False Positives in Top-50 recommendations: 7


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.